# Plotly Animated Visualizations

This notebook creates animated visualizations showing the spread of terrorism over time using Plotly.

## Outputs
1. Animated scatter_geo showing regional terrorism spread
2. Animated choropleth showing country-level incident counts

In [ ]:
import os
from pathlib import Path

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

print(f"Plotly version: {px.__version__ if hasattr(px, '__version__') else 'N/A'}")

In [ ]:
# Define paths
project_dir = Path(os.getcwd()).parent
data_file = project_dir / "data" / "processed" / "gtd_processed.parquet"
export_dir = project_dir / "exports"
export_dir.mkdir(exist_ok=True)

# Load data
df = pd.read_parquet(data_file)
print(f"Loaded {len(df):,} rows")
df.head()

## Color Schemes

In [ ]:
REGION_COLORS = {
    'Middle East & North Africa': '#e41a1c',
    'South Asia': '#377eb8',
    'Sub-Saharan Africa': '#4daf4a',
    'South America': '#984ea3',
    'Western Europe': '#ff7f00',
    'Southeast Asia': '#ffff33',
    'Eastern Europe': '#a65628',
    'North America': '#f781bf',
    'Central America & Caribbean': '#999999',
    'East Asia': '#66c2a5',
    'Central Asia': '#fc8d62',
    'Australasia & Oceania': '#8da0cb'
}

## 1. Animated Scatter Geo - Regional Terrorism Spread

In [ ]:
# Aggregate by year and region
yearly_region = df.groupby(['iyear', 'region_txt']).agg({
    'eventid': 'count',
    'latitude': 'mean',
    'longitude': 'mean',
    'nkill': 'sum',
    'nwound': 'sum'
}).reset_index()

yearly_region.columns = ['Year', 'Region', 'Incidents', 'Latitude', 'Longitude', 'Killed', 'Wounded']
yearly_region['Total_Casualties'] = yearly_region['Killed'] + yearly_region['Wounded']

print(f"Aggregated data: {len(yearly_region)} rows")
yearly_region.head()

In [ ]:
# Create animated scatter geo
fig_scatter = px.scatter_geo(
    yearly_region,
    lat='Latitude',
    lon='Longitude',
    size='Incidents',
    color='Region',
    hover_name='Region',
    hover_data={
        'Incidents': True,
        'Killed': ':.0f',
        'Wounded': ':.0f',
        'Latitude': False,
        'Longitude': False
    },
    animation_frame='Year',
    projection='natural earth',
    color_discrete_map=REGION_COLORS,
    size_max=60,
    title='Global Terrorism Spread by Region (1970-2020)'
)

fig_scatter.update_layout(
    template='plotly_dark',
    height=650,
    geo=dict(
        showland=True,
        landcolor='rgb(40, 40, 40)',
        countrycolor='rgb(60, 60, 60)',
        oceancolor='rgb(20, 20, 30)',
        showocean=True,
        showcoastlines=True,
        coastlinecolor='rgb(60, 60, 60)'
    ),
    legend=dict(
        yanchor='top',
        y=0.99,
        xanchor='left',
        x=0.01,
        bgcolor='rgba(0,0,0,0.5)'
    ),
    updatemenus=[dict(
        type='buttons',
        showactive=False,
        y=0,
        x=0.1,
        xanchor='right',
        yanchor='top',
        pad=dict(t=0, r=10),
        buttons=[dict(
            label='Play',
            method='animate',
            args=[None, dict(
                frame=dict(duration=200, redraw=True),
                fromcurrent=True,
                transition=dict(duration=100)
            )]
        )]
    )]
)

fig_scatter.show()

In [ ]:
# Export scatter geo animation
scatter_export = export_dir / "gtd_timeline_animation.html"
fig_scatter.write_html(str(scatter_export), include_plotlyjs=True, full_html=True)
print(f"Exported to: {scatter_export}")
print(f"File size: {scatter_export.stat().st_size / (1024*1024):.1f} MB")

## 2. Animated Choropleth - Country-Level Incident Counts

In [ ]:
# Country to ISO-3 mapping
COUNTRY_ISO = {
    'United States': 'USA',
    'United Kingdom': 'GBR',
    'Russia': 'RUS',
    'Germany': 'DEU',
    'France': 'FRA',
    'Spain': 'ESP',
    'Italy': 'ITA',
    'Iraq': 'IRQ',
    'Afghanistan': 'AFG',
    'Pakistan': 'PAK',
    'India': 'IND',
    'Colombia': 'COL',
    'Peru': 'PER',
    'Philippines': 'PHL',
    'Nigeria': 'NGA',
    'Somalia': 'SOM',
    'Syria': 'SYR',
    'Turkey': 'TUR',
    'Israel': 'ISR',
    'Egypt': 'EGY',
    'Algeria': 'DZA',
    'Thailand': 'THA',
    'Yemen': 'YEM',
    'Libya': 'LBY',
    'Sudan': 'SDN',
    'Bangladesh': 'BGD',
    'Sri Lanka': 'LKA',
    'Nepal': 'NPL',
    'El Salvador': 'SLV',
    'Guatemala': 'GTM',
    'Nicaragua': 'NIC',
    'Mexico': 'MEX',
    'Brazil': 'BRA',
    'Argentina': 'ARG',
    'Chile': 'CHL',
    'Greece': 'GRC',
    'Northern Ireland': 'GBR',
    'South Africa': 'ZAF',
    'Kenya': 'KEN',
    'Democratic Republic of the Congo': 'COD',
    'Cameroon': 'CMR',
    'Mali': 'MLI',
    'Niger': 'NER',
    'Burkina Faso': 'BFA',
    'Central African Republic': 'CAF',
    'Chad': 'TCD',
    'Ukraine': 'UKR',
    'Lebanon': 'LBN',
    'Iran': 'IRN',
    'Saudi Arabia': 'SAU',
    'Indonesia': 'IDN',
    'Myanmar': 'MMR',
    'Japan': 'JPN',
    'China': 'CHN',
    'Australia': 'AUS',
    'Canada': 'CAN',
    'Poland': 'POL',
    'Belgium': 'BEL',
    'Netherlands': 'NLD',
    'Sweden': 'SWE',
    'Austria': 'AUT',
    'Switzerland': 'CHE',
    'Portugal': 'PRT',
    'Ireland': 'IRL',
    'Morocco': 'MAR',
    'Tunisia': 'TUN',
    'Jordan': 'JOR',
    'Kuwait': 'KWT',
    'Bahrain': 'BHR',
    'Qatar': 'QAT',
    'United Arab Emirates': 'ARE',
    'Oman': 'OMN',
    'Venezuela': 'VEN',
    'Ecuador': 'ECU',
    'Bolivia': 'BOL',
    'Paraguay': 'PRY',
    'Uruguay': 'URY',
    'Honduras': 'HND',
    'Costa Rica': 'CRI',
    'Panama': 'PAN',
    'Cuba': 'CUB',
    'Dominican Republic': 'DOM',
    'Haiti': 'HTI',
    'Jamaica': 'JAM',
    'Trinidad and Tobago': 'TTO',
    'West Bank and Gaza Strip': 'PSE',
    'West Germany (FRG)': 'DEU',
    'East Germany (GDR)': 'DEU',
    'Soviet Union': 'RUS',
    'Yugoslavia': 'SRB',
    'Czechoslovakia': 'CZE',
    'Serbia': 'SRB',
    'Croatia': 'HRV',
    'Bosnia-Herzegovina': 'BIH',
    'Kosovo': 'XKX',
    'North Macedonia': 'MKD',
    'Montenegro': 'MNE',
    'Slovenia': 'SVN',
    'Czech Republic': 'CZE',
    'Slovakia': 'SVK',
    'Hungary': 'HUN',
    'Romania': 'ROU',
    'Bulgaria': 'BGR',
    'Albania': 'ALB',
    'Moldova': 'MDA',
    'Belarus': 'BLR',
    'Lithuania': 'LTU',
    'Latvia': 'LVA',
    'Estonia': 'EST',
    'Georgia': 'GEO',
    'Armenia': 'ARM',
    'Azerbaijan': 'AZE',
    'Kazakhstan': 'KAZ',
    'Uzbekistan': 'UZB',
    'Tajikistan': 'TJK',
    'Kyrgyzstan': 'KGZ',
    'Turkmenistan': 'TKM',
    'Vietnam': 'VNM',
    'Cambodia': 'KHM',
    'Laos': 'LAO',
    'Malaysia': 'MYS',
    'Singapore': 'SGP',
    'Brunei': 'BRN',
    'Papua New Guinea': 'PNG',
    'New Zealand': 'NZL',
    'Fiji': 'FJI',
    'South Korea': 'KOR',
    'North Korea': 'PRK',
    'Taiwan': 'TWN',
    'Hong Kong': 'HKG',
    'Mongolia': 'MNG',
    'Ethiopia': 'ETH',
    'Eritrea': 'ERI',
    'Djibouti': 'DJI',
    'Uganda': 'UGA',
    'Tanzania': 'TZA',
    'Rwanda': 'RWA',
    'Burundi': 'BDI',
    'Mozambique': 'MOZ',
    'Zimbabwe': 'ZWE',
    'Zambia': 'ZMB',
    'Malawi': 'MWI',
    'Botswana': 'BWA',
    'Namibia': 'NAM',
    'Angola': 'AGO',
    'Lesotho': 'LSO',
    'Eswatini': 'SWZ',
    'Madagascar': 'MDG',
    'Mauritius': 'MUS',
    'Senegal': 'SEN',
    'Gambia': 'GMB',
    'Guinea': 'GIN',
    'Guinea-Bissau': 'GNB',
    'Sierra Leone': 'SLE',
    'Liberia': 'LBR',
    'Ivory Coast': 'CIV',
    'Ghana': 'GHA',
    'Togo': 'TGO',
    'Benin': 'BEN',
    'Mauritania': 'MRT',
    'South Sudan': 'SSD',
    'Republic of the Congo': 'COG'
}

# Add ISO codes
df_copy = df.copy()
df_copy['iso_alpha'] = df_copy['country_txt'].map(COUNTRY_ISO)

# Check coverage
matched = df_copy['iso_alpha'].notna().sum()
print(f"Countries matched to ISO: {matched:,} / {len(df_copy):,} ({100*matched/len(df_copy):.1f}%)")

In [ ]:
# Aggregate by year and country
yearly_country = df_copy.groupby(['iyear', 'country_txt', 'iso_alpha']).agg({
    'eventid': 'count',
    'nkill': 'sum',
    'nwound': 'sum'
}).reset_index()

yearly_country.columns = ['Year', 'Country', 'ISO', 'Incidents', 'Killed', 'Wounded']
yearly_country = yearly_country.dropna(subset=['ISO'])

print(f"Yearly country data: {len(yearly_country)} rows")
yearly_country.head()

In [ ]:
# Create animated choropleth
max_incidents = yearly_country['Incidents'].quantile(0.95)

fig_choropleth = px.choropleth(
    yearly_country,
    locations='ISO',
    color='Incidents',
    hover_name='Country',
    hover_data={
        'Incidents': True,
        'Killed': ':.0f',
        'Wounded': ':.0f',
        'ISO': False
    },
    animation_frame='Year',
    color_continuous_scale='Reds',
    range_color=[0, max_incidents],
    title='Terrorist Incidents by Country (1970-2020)'
)

fig_choropleth.update_layout(
    template='plotly_dark',
    height=650,
    geo=dict(
        showland=True,
        landcolor='rgb(40, 40, 40)',
        countrycolor='rgb(60, 60, 60)',
        showcoastlines=True,
        coastlinecolor='rgb(60, 60, 60)',
        showframe=False
    ),
    coloraxis_colorbar=dict(
        title='Incidents',
        tickformat=',.0f'
    ),
    updatemenus=[dict(
        type='buttons',
        showactive=False,
        y=0,
        x=0.1,
        xanchor='right',
        yanchor='top',
        pad=dict(t=0, r=10),
        buttons=[dict(
            label='Play',
            method='animate',
            args=[None, dict(
                frame=dict(duration=200, redraw=True),
                fromcurrent=True,
                transition=dict(duration=100)
            )]
        )]
    )]
)

fig_choropleth.show()

In [ ]:
# Export choropleth animation
choropleth_export = export_dir / "gtd_choropleth_animation.html"
fig_choropleth.write_html(str(choropleth_export), include_plotlyjs=True, full_html=True)
print(f"Exported to: {choropleth_export}")
print(f"File size: {choropleth_export.stat().st_size / (1024*1024):.1f} MB")

## 3. Trend Analysis Charts

In [ ]:
# Yearly trend
yearly = df.groupby('iyear').agg({
    'eventid': 'count',
    'nkill': 'sum',
    'nwound': 'sum'
}).reset_index()
yearly.columns = ['Year', 'Incidents', 'Killed', 'Wounded']

# Create trend chart
fig_trend = make_subplots(
    rows=2, cols=1,
    subplot_titles=('Incidents per Year', 'Casualties per Year'),
    vertical_spacing=0.15
)

# Incidents
fig_trend.add_trace(
    go.Scatter(
        x=yearly['Year'],
        y=yearly['Incidents'],
        mode='lines+markers',
        name='Incidents',
        line=dict(color='#e41a1c', width=2),
        marker=dict(size=4)
    ),
    row=1, col=1
)

# Casualties
fig_trend.add_trace(
    go.Scatter(
        x=yearly['Year'],
        y=yearly['Killed'],
        mode='lines+markers',
        name='Killed',
        line=dict(color='#e41a1c', width=2),
        marker=dict(size=4)
    ),
    row=2, col=1
)

fig_trend.add_trace(
    go.Scatter(
        x=yearly['Year'],
        y=yearly['Wounded'],
        mode='lines+markers',
        name='Wounded',
        line=dict(color='#377eb8', width=2),
        marker=dict(size=4)
    ),
    row=2, col=1
)

fig_trend.update_layout(
    height=600,
    template='plotly_dark',
    title='Global Terrorism Trends (1970-2020)',
    showlegend=True,
    legend=dict(orientation='h', yanchor='bottom', y=1.02)
)

fig_trend.show()

In [ ]:
# Decade comparison
df_decade = df.copy()
df_decade['decade'] = (df_decade['iyear'] // 10) * 10
df_decade['decade_label'] = df_decade['decade'].astype(str) + 's'

decade_stats = df_decade.groupby('decade_label').agg({
    'eventid': 'count',
    'nkill': 'sum',
    'nwound': 'sum'
}).reset_index()
decade_stats.columns = ['Decade', 'Incidents', 'Killed', 'Wounded']

fig_decade = px.bar(
    decade_stats,
    x='Decade',
    y='Incidents',
    color='Incidents',
    color_continuous_scale='Reds',
    title='Incidents by Decade'
)

fig_decade.update_layout(
    template='plotly_dark',
    height=400
)

fig_decade.show()

## 4. Region Trends Over Time

In [ ]:
# Region trends
region_yearly = df.groupby(['iyear', 'region_txt']).agg({
    'eventid': 'count'
}).reset_index()
region_yearly.columns = ['Year', 'Region', 'Incidents']

fig_region_trend = px.area(
    region_yearly,
    x='Year',
    y='Incidents',
    color='Region',
    color_discrete_map=REGION_COLORS,
    title='Regional Terrorism Trends (1970-2020)'
)

fig_region_trend.update_layout(
    template='plotly_dark',
    height=500,
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=-0.3
    )
)

fig_region_trend.show()

## Summary

Created the following animated visualizations:

1. **Timeline Animation** (`gtd_timeline_animation.html`)
   - Animated scatter geo by region
   - Shows spread of terrorism over 50 years
   - Bubble size = incident count

2. **Choropleth Animation** (`gtd_choropleth_animation.html`)
   - Country-level incident counts
   - Color intensity = number of incidents
   - Play through years to see hot spots emerge

In [ ]:
# List exported files
print("Exported HTML files:")
for f in export_dir.glob("*.html"):
    size_mb = f.stat().st_size / (1024*1024)
    print(f"  {f.name}: {size_mb:.1f} MB")